## Common Words

##### corpus -> collection of all documents in a dataset (denoted by C)
##### vocabulary -> unique words that the corpus is made up of (denoted by V)
##### documents -> individual data/text entries in the corpus
##### word (or token) -> individual words inside documents in NLP

## One Hot Encoding

##### one hot encoding -> a representation technique where each word in the vocabulary is represented as a binary vector of size |V|, with 1 at the index of that word and 0 in all other positions
##### example (simple data)
##### corpus:
##### "I like NLP"
##### "I like AI"
##### vocabulary (V) = {I, like, NLP, AI}
##### one hot vectors:
##### I    -> [1, 0, 0, 0]
##### like -> [0, 1, 0, 0]
##### NLP  -> [0, 0, 1, 0]
##### AI   -> [0, 0, 0, 1]
##### pros ->
##### easy to implement
##### intuitive and simple to understand
##### works well for small vocabulary
##### cons ->
##### high sparsity (mostly zeros)
##### cannot handle out of vocabulary (OOV) word
##### no semantic meaning (AI and NLP are equally distant)
##### large memory usage for big vocabulary
##### no information about word similarity

In [4]:
from sklearn.preprocessing import OneHotEncoder
import numpy as np
import pandas as pd

In [5]:
# sample dataset
data = {
    'ID': ['D1', 'D2', 'D3', 'D4', 'D5'],
    'Relationship': [
        'user read article',
        'article read article',
        'user share article',
        'article share article',
        'user like article'
    ]
}

df = pd.DataFrame(data)
print(df)

   ID           Relationship
0  D1      user read article
1  D2   article read article
2  D3     user share article
3  D4  article share article
4  D5      user like article


In [6]:
df

,ID,Relationship
0,D1,user read article
1,D2,article read article
2,D3,user share article
3,D4,article share article
4,D5,user like article


###### in this data, vocab(v)=5 ie{user,read,share,like,article}
######  each individual row  in  df is  a  doc , so each doc(d1,d2...) in here is a size of (3,5) ie 3 words  and 5 vocab

In [7]:
# OneHotEncoder needs one word per cell, not 3 words together
# str.split(' ') separates "user read article" into 3 separate columns
# Now OneHotEncoder can encode each word individually into 5 vocab columns
words = df['Relationship'].str.split(' ', expand=True)

In [8]:
words

,0,1,2
0,user,read,article
1,article,read,article
2,user,share,article
3,article,share,article
4,user,like,article


In [9]:
encoder= OneHotEncoder(sparse_output=False)
df_encoded= encoder.fit_transform(words)


In [10]:
df_encoded

array([[0., 1., 0., 1., 0., 1.],
       [1., 0., 0., 1., 0., 1.],
       [0., 1., 0., 0., 1., 1.],
       [1., 0., 0., 0., 1., 1.],
       [0., 1., 1., 0., 0., 1.]])

In [11]:
feature_names = encoder.get_feature_names_out()

In [12]:
feature_names

array(['x0_article', 'x0_user', 'x1_like', 'x1_read', 'x1_share',
       'x2_article'], dtype=object)

In [13]:
df=pd.DataFrame(df_encoded, columns=feature_names)

In [14]:
df

,x0_article,x0_user,x1_like,x1_read,x1_share,x2_article
0,0.0,1.0,0.0,1.0,0.0,1.0
1,1.0,0.0,0.0,1.0,0.0,1.0
2,0.0,1.0,0.0,0.0,1.0,1.0
3,1.0,0.0,0.0,0.0,1.0,1.0
4,0.0,1.0,1.0,0.0,0.0,1.0


Explanation

**Step 1: Original data**
```
Relationship
user read article        ← D1
article read article     ← D2
user share article       ← D3
article share article    ← D4
user like article        ← D5
```

**Step 2: After split into 3 columns**
```
Word1      Word2    Word3
user       read     article
article    read     article
user       share    article
article    share    article
user       like     article
```

**Step 3: OneHotEncoder creates columns for each category**
- **Word1 categories:** {article, user} → x0_article, x0_user
- **Word2 categories:** {like, read, share} → x1_like, x1_read, x1_share
- **Word3 categories:** {article} → x2_article

**Step 4: Encode (1.0 if match, 0.0 if not)**
```
D1: user, read, article     → x0_user=1.0, x1_read=1.0, x2_article=1.0
D2: article, read, article  → x0_article=1.0, x1_read=1.0, x2_article=1.0
D3: user, share, article    → x0_user=1.0, x1_share=1.0, x2_article=1.0
D4: article, share, article → x0_article=1.0, x1_share=1.0, x2_article=1.0
D5: user, like, article     → x0_user=1.0, x1_like=1.0, x2_article=1.0
```


## Bag of Words

# Bag-of-Words (BoW) – Intuition & Workflow

**Step 1:** Form the whole corpus and create a vocabulary of all unique words.  

**Step 2:** For each document, count how many times each word from the vocabulary appears.  

**Step 3:** Represent the document as a vector of word counts (or binary presence/absence).  


**Important Note:**  
In Bag-of-Words (BoW), **the order of words and context does NOT matter**.  
- Only the **frequency or presence** of words is considered.  
- For example, the sentences:  
  - "I love NLP"  
  - "NLP love I"  

  Will have the **same BoW representation** because they contain the same words, even though the order is different.

## How Semantic Meaning is Captured

- Bag-of-Words **does NOT capture semantic meaning** directly.  
  - It only considers **which words appear** and **how often**, ignoring order and context.  

- To capture **meaning and relationships between words**, we need:  
  1. **Word Embeddings** (e.g., Word2Vec, GloVe)  
     - Words with similar meaning have **vectors close together** in embedding space.  
  2. **Contextual Embeddings** (e.g., BERT, GPT)  
     - Words are represented based on **their context in the sentence**, capturing subtle nuances.  

**Core idea:**  
> Unlike BoW, embedding-based methods encode **semantics**, so "king" and "queen" are recognized as related, while BoW treats them as completely independent features.

**Core Intuition:**  
Documents with similar content will have similar word distributions, so their vectors will be close in representation space. This helps algorithms detect similarity and classify text effectively.

In [19]:
# sample dataset
data = {
    'ID': ['D1', 'D2', 'D3', 'D4', 'D5'],
    'Relationship': [
        'user read article',
        'article read article',
        'user share article',
        'article share article',
        'user like article'
    ]
}

df = pd.DataFrame(data)
print(df)

   ID           Relationship
0  D1      user read article
1  D2   article read article
2  D3     user share article
3  D4  article share article
4  D5      user like article


In [20]:
from sklearn.feature_extraction.text import CountVectorizer

In [29]:
cv=CountVectorizer()
bow=cv.fit_transform(df["Relationship"])

In [30]:
bow

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 13 stored elements and shape (5, 5)>

In [31]:
#vocabulary
print(cv.vocabulary_)

{'user': 4, 'read': 2, 'article': 0, 'share': 3, 'like': 1}



The columns are ordered by these indices:
```
Index:  0        1      2       3       4
Word:  article  like  read  share  user
```


In [36]:
d1_output=words[0].toarray()

In [38]:
d1_output

array([[1, 0, 1, 0, 1]])

## How Encoding Works

Each document is converted to a binary vector (1 = word present, 0 = word absent) following the vocabulary order.


### Example 1: D1 - "user read article"

Check each index position in the vocabulary:
```
Index 0 (article): 1 ✓ present in D1
Index 1 (like):    0 ✗ NOT present in D1
Index 2 (read):    1 ✓ present in D1
Index 3 (share):   0 ✗ NOT present in D1
Index 4 (user):    1 ✓ present in D1
```

**Output: [1, 0, 1, 0, 1]** ✓

In [42]:
d2_output=words[1].toarray()

In [43]:
d2_output

array([[2, 0, 1, 0, 0]])

## Example 2: D2 - "article read article"

Check each index position:
```
Index 0 (article): 2 ✓ present in D2(cause  article is  repeated  twice)
#note: CountVectorizer counts how many times each word appears
Index 1 (like):    0 ✗ NOT present in D2
Index 2 (read):    1 ✓ present in D2
Index 3 (share):   0 ✗ NOT present in D2
Index 4 (user):    0 ✗ NOT present in D2
```

**Output: [2, 0, 1, 0, 0]** ✓


In [44]:
d3_output=words[2].toarray()

In [46]:
d3_output

array([[1, 0, 0, 1, 1]])

"user share article" ie -> index 1 ie article ->present  so 1, then "like" at index 2  and "read" are not present so 0 0 and "share" is  present so  1,and user is present so 1.
ie 10011

In [47]:
cv.transform([" user read and write article"]).toarray()

array([[1, 0, 1, 0, 1]])

In [ ]:
##  for new sentence, if we see, "and" and "write"  were not during  training, so they are not shown here.
## oov(out  of vocabulary) problem doesnt  occur here

## Limitations of Bag-of-Words (BoW)
 **Loses word order entirely**  
Example: `"people watch campusx"` and `"campusx watch people"` are treated identically.  
Cannot distinguish between `"dog bites man"` vs `"man bites dog"`.

 **No context captured**  
BoW only counts word frequencies: `{people:1, watch:1, campusx:1}`  
It cannot understand relationships or meaning between words.

 **Sparsity problem**  
As vocabulary grows, the feature representation becomes sparse.  
Feature vectors contain lots of zeros, which can be inefficient for computation.

##  N-Grams (Bag of n-grams)

n-gram is a contiguous sequence of n items (usually words or characters) from a given text. They are used to capture local context and word order, unlike basic Bag-of-Words.

# Bigrams Representation

## Step 1: Extract All Bigrams

#### D1: "people watch campusx"
- Bigrams: **(people, watch)**, **(watch, campusx)**

#### D2: "campusx watch campusx"
- Bigrams: **(campusx, watch)**, **(watch, campusx)**

#### D3: "people write comment"
- Bigrams: **(people, write)**, **(write, comment)**

#### D4: "campusx write comment"
- Bigrams: **(campusx, write)**, **(write, comment)**

---

## Step 2: Bigram Vocabulary (All Unique Bigrams)

| ID | Bigram |
|---|---|
| BG1 | (people, watch) |
| BG2 | (watch, campusx) |
| BG3 | (campusx, watch) |
| BG4 | (people, write) |
| BG5 | (write, comment) |
| BG6 | (campusx, write) |

#### Total bigram vocabulary size: 6

---

## Step 3: Bigram Representation Matrix

|  | BG1 (people, watch) | BG2 (watch, campusx) | BG3 (campusx, watch) | BG4 (people, write) | BG5 (write, comment) | BG6 (campusx, write) |
|---|---|---|---|---|---|---|
| **D1** | 1 | 1 | 0 | 0 | 0 | 0 |
| **D2** | 0 | 1 | 1 | 0 | 0 | 0 |
| **D3** | 0 | 0 | 0 | 1 | 1 | 0 |
| **D4** | 0 | 0 | 0 | 1 | 1 | 1 |

---
## Key Observations

####  D1 vs D2 are now different!
- D1 has BG1, but D2 has BG3
- We now distinguish between "people watch" and "campusx watch"

####  Sparsity is high
- 18 cells, only 6 contain 1s (67% sparse)
- Vocabulary size: 6 features (for just 4 short documents!)

####  Order matters now
- (people, watch) ≠ (watch, people)
- Can't represent reversed pairs without explicit feature

####  Still limitations:
- Can't capture that D3 and D4 are similar (both write-comment)
- Missing single-word "write" frequency
- Data grows sparsely with larger vocabularies

# Trigrams Representation

## Step 1: Extract All Trigrams

#### D1: "people watch campusx"
- Trigrams: **(people, watch, campusx)**

#### D2: "campusx watch campusx"
- Trigrams: **(campusx, watch, campusx)**

#### D3: "people write comment"
- Trigrams: **(people, write, comment)**

#### D4: "campusx write comment"
- Trigrams: **(campusx, write, comment)**

---


## Step 2: Trigram Vocabulary (All Unique Trigrams)

| ID | Trigram |
|---|---|
| TG1 | (people, watch, campusx) |
| TG2 | (campusx, watch, campusx) |
| TG3 | (people, write, comment) |
| TG4 | (campusx, write, comment) |

#### Total trigram vocabulary size: 4

### each sentence is basically a tri-gram
---

## Step 3: Trigram Representation Matrix

|  | TG1 (people, watch, campusx) | TG2 (campusx, watch, campusx) | TG3 (people, write, comment) | TG4 (campusx, write, comment) |
|---|---|---|---|---|
| **D1** | 1 | 0 | 0 | 0 |
| **D2** | 0 | 1 | 0 | 0 |
| **D3** | 0 | 0 | 1 | 0 |
| **D4** | 0 | 0 | 0 | 1 |

---

## Key Observations

#### ✓ Complete context captured!
- Each document is now represented by its complete phrase
- Full sentence structure is preserved
- D1, D2, D3, D4 are all completely distinct

#### ✓ Perfect separation
- Each document maps to exactly ONE unique trigram
- No overlap between documents
- Clear distinction even between similar patterns

#### Extreme sparsity problem!
- Matrix is 80% sparse (only 4 ones out of 16 cells)
- Vocabulary: 4 features for 4 documents
- This ratio gets much worse with larger datasets

#### Scalability issues:
- With vocabulary V, trigram combinations = V³
- If vocabulary size = 100 words → 1,000,000 possible trigrams
- If vocabulary size = 1,000 words → 1 billion possible trigrams!
- Most trigrams appear only once (data sparsity explosion)

#### Loss of generalization:
- Trigrams are TOO specific
- Can't find patterns across documents
- "people X comment" and "campusx X comment" are treated as completely different
- No semantic similarity captured

---

## Comparison: Bigrams vs Trigrams

| Aspect | Bigrams | Trigrams |
|---|---|---|
| **Vocabulary Size** | 6 | 4 |
| **Sparsity** | 67% | 80% |
| **Context Captured** | Moderate (2 words) | Complete (3 words) |
| **Generalization** | Better | Worse |
| **Computational Cost** | Lower (V²) | Higher (V³) |
| **Documents Distinguishable** | Mostly | Perfectly (too specific) |

---

## Why Higher N-grams Fail

As n increases:
- ✓ More context captured
- ✗ Vocabulary explodes exponentially (V^n)
- ✗ Sparsity increases dramatically
- ✗ Overfitting to training data
- ✗ Poor generalization to new text
- ✗ Computational complexity unbearable

**This is why modern NLP uses neural embeddings (Word2Vec, BERT, etc.) instead of high-order n-grams!**

In [62]:
import pandas as pd

# Create the dataframe with 'Document' as the index
df = pd.DataFrame(
    {
        'Text': [
            'people watch campusx',
            'campusx watch campusx',
            'people write comment',
            'campusx write comment'
        ]
    },
    index=['D1', 'D2', 'D3', 'D4']
)

df

,Text
D1,people watch campusx
D2,campusx watch campusx
D3,people write comment
D4,campusx write comment


##### Using n_grams in text vectorization
##### we use ngram_range=(,)
####
##### Examples:
##### (2, 2) → creates only bi-grams (2-word sequences)
##### (3, 3) → creates only tri-grams (3-word sequences)
##### (1, 2) → creates both uni-grams and bi-grams
##### (2, 3) → creates both bi-grams and tri-grams

In [69]:
from sklearn.feature_extraction.text import CountVectorizer
cv=CountVectorizer(ngram_range=(2,2)) #creates  bi-grams

In [70]:
bigrams=cv.fit_transform(df["Text"])

In [71]:
#vocab
print(cv.vocabulary_)

{'people watch': 2, 'watch campusx': 4, 'campusx watch': 0, 'people write': 3, 'write comment': 5, 'campusx write': 1}


In [72]:
print(bigrams[0].toarray())

[[0 0 1 0 1 0]]


In [74]:
print(bigrams[1].toarray())

[[1 0 0 0 1 0]]


In [75]:
print(bigrams[3].toarray())

[[0 1 0 0 0 1]]


###### for tri-grams, we use ngram_range=(3,3)..and on as explained above